In [1]:
import pandas as pd
import numpy as np

In [30]:
        
data = pd.read_excel(r"/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/2022-12-SGFTFN_CARBONATES_unprocessed.xlsx",header=0,index_col=0)
# data = pd.concat([data,data1],axis=0)
# data = pd.read_excel(r"/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/2022-12-SGFTFN_CLAY_MINERALS_unprocessed.xlsx",header=0,index_col=0)

oxide = pd.read_excel(r"/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/oxide_data.xlsx", sheet_name="Sheet1",index_col=0,header=0)
oxlist = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5",'CO2','SrO','BaO']

def wt_to_mol(data, oxide = oxide):
    data[data<=2] = 0
    oxlist1 = oxlist.copy()
    oxide = oxide.T[oxlist1].iloc[0, :].to_numpy()
    data = normalize(data)
    [r, c] = data.shape
    data_f = np.empty((r, c))
    for i in range(0, c):
        data_f[:, i] = data[:, i] / oxide[i]
    data
    data_f = normalize(data_f)
    return(data_f.round(2))


def normalize(data):
    [r, c] = data.shape
    a = data.sum(axis=1).reshape((len(data), 1))
    data_formatted = ((data*100)/a).round(1)
    return(data_formatted)


def cat_calc(data1,oxide_list):
    total = data1.columns.get_loc("Total")
    o_no = data1.loc[:,"Oxygen_no"]
    data = data1.iloc[:,:total].copy()
    oxide = oxide_list.loc[data.columns]
    data = data.div(oxide['Mol. Wt.'].values,axis=1).round(3)
    data = data.mul(oxide['O_no'].values,axis=1).round(3)
    norm = o_no.div(data.sum(axis=1)).round(3)
    data = data.mul(norm.values,axis=0).round(3)
    data = data.mul(oxide['Cat_per_o'].values,axis=1).round(3)
    total = data.sum(axis=1).round(3)
    data.columns = oxide['Cation'].values
    data['Cation_Total'] = total
    return data.round(3)


In [31]:
data2 = data[['SIO2(WT%)','TIO2(WT%)','AL2O3(WT%)','CR2O3(WT%)','FE2O3T(WT%)', 'FE2O3(WT%)', 'FEOT(WT%)','FEO(WT%)', 'MNO(WT%)','MGO(WT%)','CAO(WT%)', 'NA2O(WT%)','K2O(WT%)','P2O5(WT%)','CO2(WT%)','SRO(WT%)','BAO(WT%)','MINERAL']]

In [32]:
data2.MINERAL.unique()

array(['CARBONATE', 'CALCITE', 'ARAGONITE', 'MAGNESITE', 'DOLOMITE',
       'SIDERITE', 'ANKERITE', 'NYEREREITE', 'GREGORYITE', 'SPURRITE',
       'SHORTITE', 'RHODOCHROSITE', 'MALACHITE', 'STRONTIANITE',
       'ANCYLITE', 'CARBOCERNAITE', 'KHANNESHITE', 'PIRSSONITE',
       'BARYTOCALCITE', 'NORSETHITE', 'BREUNNERITE', 'MAGNESIOSIDERITE',
       'BURBANKITE', 'CORDYLITE', 'CEBAITE', 'MCKELVEYITE',
       'DOLOMITE-ANKERITE', 'KUTNAHORITE', 'KUKHARENKOITE',
       'FLUORO-CARBONATE', 'QAQARSSUKITE', 'HUANGHOITE', 'WITHERITE',
       'OLEKMINSKITE', 'ALSTONITE', nan], dtype=object)

In [44]:
data_cleaned = data2.loc[~data2['CO2(WT%)'].isna(),:]
# data_px = data_cleaned[(data_cleaned["MINERAL"]=="MAGNETITE") | (data_cleaned["MINERAL"]=="TITANO-MAGNETITE")]
data_px = data_cleaned.copy()
data_px = data_px.iloc[:,:-1].apply(pd.to_numeric,args=('coerce',)).astype('float')

cond = (data_px['FEOT(WT%)'].isna()) & (~data_px['FE2O3T(WT%)'].isna())
data_px.loc[cond,'FEOT(WT%)'] = data_px.loc[cond,'FE2O3T(WT%)']*.8998
cond = (data_px['FEOT(WT%)'].isna()) & (data_px['FE2O3T(WT%)'].isna())
data_px.loc[cond,'FEOT(WT%)'] = data_px.loc[cond,'FEO(WT%)'] + (data_px.loc[cond,'FE2O3(WT%)']*.8998)
data_px.loc[~data_px['FEOT(WT%)'].isna(),:]
data_px.pop("FE2O3(WT%)")
data_px.pop("FE2O3T(WT%)")
data_px.pop("FEO(WT%)")
data_px['Mineral'] = data_cleaned['MINERAL']
data_px.columns = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O","P2O5",'CO2','SrO','BaO','Mineral']
mineral = data_px['Mineral']
data_px = data_px.iloc[:,:-1].apply(pd.to_numeric,args=('coerce',)).astype('float')
data_px2 = data_px.fillna(0)
total = data_px2.sum(axis=1)
data_px2['Mineral'] = mineral
data_px2 = data_px2[(total>=85) & (total<=101)]
mineral = data_px2.pop("Mineral")
col = data_px2.columns
ind = data_px2.index
data_px2 = pd.DataFrame(wt_to_mol(data_px2.to_numpy(),oxide),columns = col,index=ind)
# data_px2 = data

non_essential_sum = data_px2[['SiO2','TiO2','Al2O3',"TiO2", "Na2O", "K2O","P2O5"]].sum(axis=1)
data_px2 = data_px2[non_essential_sum<=5]
m = data_px2[["FeO","MnO","MgO",'CaO','SrO','BaO']].sum(axis=1)
data_px2 = data_px2[(data_px2.CO2 <= 51) & (m >=49) ]

# data_px2 = data_px2[ (m >= 30) & (m <= 100)]
# data_px2 = data_px2[(data_px2.Al2O3 + data_px2.Cr2O3 >= 49) & (data_px2.Al2O3 + data_px2.Cr2O3 <= 51)]
# data_px2['P2O5'] = 0
data_px2['Mineral'] = "Cb"
data_px2['Al2O3'] = data_px2['Al2O3'] + data_px2['Cr2O3']
data_px2.pop("Cr2O3")
data_px2['CaO'] = data_px2[['CaO','BaO','SrO']].sum(axis=1)
data_px2 = data_px2.drop(columns=['SrO','BaO'])
data_px2.to_excel("/Users/aditya/Documents/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/molar tables/new data/Carbonate_processed_mol.xlsx")
data_px2['M'] = data_px2[["FeO","MnO","MgO"]].sum(axis=1)
data_px2.sort_values("CO2",ascending=False)


,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,CO2,Mineral,M
CITATION,,,,,,,,,,,,,
[21228] GUARINO V. (2017),0.0,0.0,0.0,0.0,0.0,23.1,25.9,0.0,0.0,0.0,51.0,Cb,23.1
[19521] XU CHENG (2015),0.0,0.0,0.0,0.0,3.5,0.0,45.5,0.0,0.0,0.0,51.0,Cb,3.5
[24093] HEGNER E. (2020),0.0,0.0,0.0,0.0,0.0,0.0,49.0,0.0,0.0,0.0,51.0,Cb,0.0
[23632] TOSCANI L. (2020),0.0,0.0,0.0,0.0,0.0,0.0,49.0,0.0,0.0,0.0,51.0,Cb,0.0
[17223] ZAITSEV A. N. (2013),0.0,0.0,0.0,0.0,0.0,0.0,49.0,0.0,0.0,0.0,51.0,Cb,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
[18147] AZER M. K. (2010),0.0,0.0,0.0,3.4,0.0,29.2,22.4,0.0,0.0,0.0,44.9,Cb,32.6
[23926] NOSOVA A. A. (2020),0.0,0.0,0.0,0.0,0.0,2.5,53.1,0.0,0.0,0.0,44.4,Cb,2.5
[18925] NYKƒNEN J. (1997),0.0,0.0,0.0,0.0,0.0,0.0,55.7,0.0,0.0,0.0,44.3,Cb,0.0
